## Phase relation Comparison in NoRad and CldRad

### Import package

In [50]:
import h5py 
import numpy as np
import pandas as pd

from typing import cast
from pathlib import Path
from scipy.signal import correlate, correlation_lags

from matplotlib import pyplot as plt

# using style sheet
style_path = Path("/home/b11209013/Kuang2008_v0.3.0_Analysis/style_sheet/SingleLine.mplstyle")
plt.style.use(["seaborn-v0_8-colorblind", str(style_path)])

### Helper functions

In [51]:
def phase_shifts(
        field1: np.ndarray,
        field2: np.ndarray,        
) -> np.ndarray:
    
	# dimension check
	if field1.shape != field2.shape:
		raise ValueError("Input signals must have  the same shape.")

	# shapes
	nz, nphase = field1.shape

	# calculate phase difference
	d_phase: np.ndarray = 2*np.pi / (nphase-1)

	# Calculate lages
	lags: np.ndarray = correlation_lags(nphase, nphase, mode="full")

	phase_shift: np.ndarray = np.zeros(nz)

	for i in range(nz):
		# caluclate cross-correlation
		corr: np.ndarray = correlate(field1[i], field2[i], mode="full")

		# Find index of maximum correlation
		max_corr_idx: int = np.nanargmax(corr).astype(int)

		# convert index shift to phase shift
		phase_shift[i] = lags[max_corr_idx] * d_phase
	
	return phase_shift

### Load data

In [52]:
# Path for input file
NoPath: Path = Path("/home/b11209013/Kuang2008_v0.3.0_Analysis/Files/NoRad/Rolled")
CldPath: Path = Path("/home/b11209013/Kuang2008_v0.3.0_Analysis/Files/CldRad/Rolled")

# Load NoRad data
NoState: dict[str, np.ndarray] = {
    "J": np.array(pd.read_csv(NoPath / "J.csv"))[:, 1:],
    "T": np.array(pd.read_csv(NoPath / "T.csv"))[:, 1:],
    "w": np.array(pd.read_csv(NoPath / "w.csv"))[:, 1:]
}

CldState: dict[str, np.ndarray] = {
    "J": np.array(pd.read_csv(CldPath / "J.csv"))[:, 1:],
    "T": np.array(pd.read_csv(CldPath / "T.csv"))[:, 1:],
    "w": np.array(pd.read_csv(CldPath / "w.csv"))[:, 1:]
}

### Calculate phase relations

In [53]:
# Calculate Q-T phase relation
No_QT_shift : np.ndarray = phase_shifts(NoState["T"], NoState["J"])
Cld_QT_shift: np.ndarray = phase_shifts(CldState["T"], CldState["J"])

# Calculate w-T phase relation
No_wT_shift : np.ndarray = phase_shifts(NoState["T"], NoState["w"])
Cld_wT_shift: np.ndarray = phase_shifts(CldState["T"], CldState["w"])

### Visualization

#### Q, T phase relation

In [56]:
fig, ax = plt.subplots(1, 1, figsize=(6, 9))

ax.plot(
    No_QT_shift[1:],
    np.linspace(0, 14000, No_QT_shift.shape[0])[1:],
    label="No Rad.", linewidth=3
    )
ax.plot(
    Cld_QT_shift[1:],
    np.linspace(0, 14000,Cld_QT_shift.shape[0])[1:],
    label="Cloud Rad.", linewidth=3
    )
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.minorticks_on()
ax.set_xlim(-np.pi, np.pi)
ax.set_ylim(0, 14000)
ax.set_xticks(np.linspace(-np.pi, np.pi, 5), [r"-$\pi$", r"-$\pi$/2", r"0", r"$\pi$/2", r"$\pi$"])
ax.set_yticks(np.linspace(0, 14000, 8))
ax.set_xlabel("Phase [rad]")
ax.set_ylabel("Level [m]")
ax.legend(frameon=False, loc="best")
plt.savefig("/home/b11209013/Kuang2008_v0.3.0_Analysis/Figure/NoRad_vs_CldRad/QT_phase.png", dpi=600, bbox_inches="tight")
plt.close(fig)

#### w, T phase relation

In [57]:
fig, ax = plt.subplots(1, 1, figsize=(6, 9))

ax.plot(
    No_wT_shift[1:],
    np.linspace(0, 14000, No_wT_shift.shape[0])[1:],
    label="No Rad.", linewidth=3
    )
ax.plot(
    Cld_wT_shift[1:],
    np.linspace(0, 14000, Cld_wT_shift.shape[0])[1:],
    label="Cloud Rad.", linewidth=3
    )
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.minorticks_on()
ax.set_xlim(-np.pi, np.pi)
ax.set_ylim(0, 14000)
ax.set_xticks(np.linspace(-np.pi, np.pi, 5), [r"-$\pi$", r"-$\pi$/2", r"0", r"$\pi$/2", r"$\pi$"])
ax.set_yticks(np.linspace(0, 14000, 8))
ax.set_xlabel("Phase [rad]")
ax.set_ylabel("Level [m]")
ax.legend(frameon=False, loc="best")
plt.savefig("/home/b11209013/Kuang2008_v0.3.0_Analysis/Figure/NoRad_vs_CldRad/wT_phase.png", dpi=600, bbox_inches="tight")
plt.close(fig)